In [1]:
# ¿Cómo preparamos los abstracts de Scopus para identificar sus términos distintivos sin que textos vacíos, fórmulas retóricas y avisos de copyright distorsionen la representación?

import pandas as pd

dataframe = pd.read_csv("../data/scopus_abstracts.csv.gz", compression="gzip")
dataframe[["scopus_record_id", "publication_title", "abstract"]].head()

,scopus_record_id,publication_title,abstract
0,2-s2.0-105042759102,Strategic Pathways in Malaysia’s Fintech Secto...,This study examines the relationship between e...
1,2-s2.0-85175999008,"ELI Principles on Blockchain Technology, Smart...",The project started in 2018 following a Use-Ca...
2,2-s2.0-85149119289,Bitland-A Decentralized Commercial Real Estate...,Real estate has always been a significant inve...
3,2-s2.0-84931062324,"The politics of person, property, technology: ...",Global policy organizations have emphasized ca...
4,2-s2.0-85049126196,Planned preventative maintenance and proptech,[No abstract available]


In [2]:
# Los abstracts muy breves suelen ser marcadores de ausencia y no aportan contenido para una representación documento–término.

dataframe["abstract_word_count"] = dataframe["abstract"].str.split().str.len()
dataframe.sort_values("abstract_word_count")[
    ["abstract_word_count", "abstract"]
].head(20)

,abstract_word_count,abstract
203,3,[No abstract available]
375,3,[No abstract available]
39,3,[No abstract available]
188,3,[No abstract available]
342,3,[No abstract available]
106,3,[No abstract available]
219,3,[No abstract available]
175,3,[No abstract available]
25,3,[No abstract available]
328,3,[No abstract available]


In [3]:
# Se excluyen marcadores conocidos de ausencia y textos demasiado cortos antes de transformar sus palabras.

minimum_abstract_words = 20
missing_abstract_markers = {"[no abstract available]"}

dataframe["has_missing_abstract_marker"] = dataframe["abstract"].str.casefold().isin(
    missing_abstract_markers
)
dataframe["has_sufficient_content"] = (
    ~dataframe["has_missing_abstract_marker"]
    & dataframe["abstract_word_count"].ge(minimum_abstract_words)
)
excluded_abstracts = int((~dataframe["has_sufficient_content"]).sum())
dataframe = dataframe.loc[dataframe["has_sufficient_content"]].copy()
dataframe["normalized_text"] = dataframe["abstract"].str.lower()

excluded_abstracts, len(dataframe)

(20, 385)

In [4]:
# Los finales alineados a la derecha y ordenados de derecha a izquierda agrupan avisos editoriales similares.

def print_aligned_text_tails(texts, tail_word_count=15, tail_sample_size=20):
    evidence = pd.DataFrame(
        {
            "text_tail": texts.str.split().str[-tail_word_count:].str.join(" "),
            "reversed_text": texts.str[::-1],
        }
    ).sort_values("reversed_text")
    text_tails = evidence["text_tail"].head(tail_sample_size)
    tail_width = text_tails.str.len().max()

    for text_tail in text_tails:
        print(text_tail.rjust(tail_width))


print_aligned_text_tails(dataframe["normalized_text"])

                                          has been considered the last human frontier in the technological world. © 2023 the author(s)
                                       that can be used alongside the classic world city connectivity indicators. © 2023 the author(s)
                                     the nigerian environment. the implication of key findings is also discussed. © 2024 the author(s)
                                 igi global scientific publishing. all rights reserved. no part of this publication may be reproduced,
                                             new figure displaccer, paet, citytim = proptech platforms and smart city systems © 2026 .
                              with gm foods and recombinant technology. © 2012 association of food scientists & technologists (india).
                          as it has not been much explained. © 2019 international business information management association (ibima).
the original work is attributed as specified on the sag

In [5]:
# Los avisos editoriales al final no describen el contenido del artículo y añadirían términos irrelevantes al vocabulario.

copyright_pattern = r"\s*(?:(?:copyright\s*)?©|copyright\b|all rights reserved\b).*?$"
dataframe["has_copyright_notice"] = dataframe["normalized_text"].str.contains(
    copyright_pattern,
    case=False,
    regex=True,
)
dataframe["has_copyright_notice"].sum()

335

In [6]:
# Se elimina el aviso editorial antes de conservar una versión del abstract orientada al contenido.

dataframe["normalized_text"] = dataframe["normalized_text"].str.replace(
    copyright_pattern,
    "",
    case=False,
    regex=True,
)

In [7]:
# La misma evidencia permite verificar que el reemplazo retiró los finales editoriales repetidos.

print_aligned_text_tails(dataframe["normalized_text"])

              obtained, the proposed methodology tends to be efficient in terms of privacy protection, receiver anonymity,
                                                when x was 5% and it had the highest remanent polarization when x was 15%.
                academics, property students and senior school executives (e.g. deputy dean and deputy head of education).
                         a lot in the transformation of traditional real estate firms into digitally native brands (dnbs).
bim-gis semi-automatic prototype based on the depreciated reconstruction cost through an experimentation in rende (italy).
            also included in the business & management collection which can be accessed at https://hstalks.com/ business/.
          is also included in the business & management collection which can be accessed at https://hstalks.com/business/.
          is also included in the business & management collection which can be accessed at https://hstalks.com/business/.
                

In [8]:
# Se presentan conectores discursivos de varias palabras que aparecen en el corpus; son expresiones generales del inglés y no fórmulas retóricas de abstracts.

selected_connectors = [
    "as a consequence",
    "as a result",
    "as well as",
    "for example",
    "for instance",
    "in addition",
    "in conclusion",
    "in fact",
    "in other words",
    "in particular",
    "in summary",
    "on the other hand",
]

connector_counts = pd.DataFrame(
    {
        "connector": selected_connectors,
        "occurrences": [
            dataframe["normalized_text"].str.count(r"\b" + connector + r"\b").sum()
            for connector in selected_connectors
        ],
    }
).query("occurrences > 0")

connector_counts.sort_values("occurrences", ascending=False)

,connector,occurrences
2,as well as,43
5,in addition,30
9,in particular,15
1,as a result,12
3,for example,9
11,on the other hand,7
4,for instance,5
6,in conclusion,2
7,in fact,1


In [9]:
# Se marcan los conectores sin eliminarlos todavía para que la etapa posterior pueda decidir si conservan o no valor analítico.

for connector in sorted(selected_connectors, key=len, reverse=True):
    dataframe["normalized_text"] = dataframe["normalized_text"].str.replace(
        r"\b" + connector + r"\b",
        connector.replace(" ", "_"),
        regex=True,
    )

abstract_previews = dataframe["normalized_text"].str[:130]
marked_abstracts = abstract_previews[abstract_previews.str.contains("_")].head(8)

for abstract in marked_abstracts:
    print(abstract)

the structural and optoelectronic properties, as_well_as epitaxial growth peculiarities of thin film strained layer si1-xgex/si he
the industry in general and the environment in_particular have benefited enormously by utilizing electrostatic precipitators to re
the volume and complexity of publicly available real estate data have been snowballing. as_a_result, information extraction and pr
purpose: the purpose of this paper is to stimulate researchers’ understanding of place in general and psychogeography in_particula
benzo[a]pyrene (bap) is a high molecular weight polycyclic aromatic hydrocarbon produced as_a_result of incomplete combustion of o
purpose: due to the young age of proptech, little is known about the dynamics of its expansion. in_particular, there is limited ag
artificial intelligence (ai) technologies have developed rapidly, and generative ai in_particular challenges human creativity. the


In [10]:
# Se presentan fórmulas retóricas de abstracts que aparecen en el corpus; no son conectores discursivos y se tratarán en una etapa distinta.

rhetorical_scaffolding = [
    "in this article",
    "in this paper",
    "in this study",
    "in this work",
    "is based on",
    "little is known about",
    "many studies",
    "the article examines",
    "the article explores",
    "in the field of",
]

scaffolding_counts = pd.DataFrame(
    {
        "formula": rhetorical_scaffolding,
        "occurrences": [
            dataframe["normalized_text"].str.count(r"\b" + formula + r"\b").sum()
            for formula in rhetorical_scaffolding
        ],
    }
).query("occurrences > 0")

scaffolding_counts.sort_values("occurrences", ascending=False)

,formula,occurrences
1,in this paper,26
2,in this study,12
0,in this article,8
9,in the field of,8
4,is based on,7
3,in this work,4
5,little is known about,4
8,the article explores,2
6,many studies,1
7,the article examines,1


In [11]:
# Se marcan las fórmulas retóricas para evidenciar que son unidades distintas antes de decidir su eliminación.

for formula in sorted(rhetorical_scaffolding, key=len, reverse=True):
    dataframe["normalized_text"] = dataframe["normalized_text"].str.replace(
        r"\b" + formula + r"\b",
        formula.replace(" ", "_"),
        regex=True,
    )

scaffolding_markers = [formula.replace(" ", "_") for formula in rhetorical_scaffolding]
scaffolding_pattern = "|".join(scaffolding_markers)
abstract_previews = dataframe["normalized_text"].str[:130]
marked_abstracts = abstract_previews[abstract_previews.str.contains(scaffolding_pattern)].head(8)

for abstract in marked_abstracts:
    print(abstract)

in_this_article by the editors of this special issue, the authors discuss the changes being brought to real estate investment by n
blockchain is a versatile technology with many potential applications, and one of them is in_the_field_of real estate. blockchain 
purpose: many_studies have analysed the impact of various variables on the ability of companies to raise capital. while most of th
with the emergence of property technology, online listing data have drawn increasing interest in_the_field_of real estate–related 
in_this_study, it is planned that the volume-2 will provide technical information on thematic issues related to artificial intelli
the success of digital transformation in real estate (re) and facility management (fm) is_based_on understanding modern digitalisa
in_this_study, we focus on the innovation resistance that has appeared in the process of accepting changed information technology 
our previous research was focused on reactive powder concrete (rpc). in_this_articl

In [12]:
# Las fórmulas marcadas no distinguen los temas del corpus, por lo que se eliminan antes de tokenizar.

for marker in scaffolding_markers:
    dataframe["normalized_text"] = dataframe["normalized_text"].str.replace(
        r"\b" + marker + r"\b",
        " ",
        regex=True,
    )

dataframe["normalized_text"] = dataframe["normalized_text"].str.split().str.join(" ")

In [13]:
# Se separa cada abstract en tokens para que el texto se convierta en unidades procesables.

from nltk.tokenize import word_tokenize

dataframe["tokens"] = dataframe["normalized_text"].map(word_tokenize)
dataframe[["normalized_text", "tokens"]].head()

,normalized_text,tokens
0,this study examines the relationship between e...,"[this, study, examines, the, relationship, bet..."
1,the project started in 2018 following a use-ca...,"[the, project, started, in, 2018, following, a..."
2,real estate has always been a significant inve...,"[real, estate, has, always, been, a, significa..."
3,global policy organizations have emphasized ca...,"[global, policy, organizations, have, emphasiz..."
5,coffee is arguably the most popular drink in t...,"[coffee, is, arguably, the, most, popular, dri..."


In [14]:
# Se conservan tokens alfabéticos y se hace visible la información que la regla descarta.

dataframe["tokens"] = dataframe["tokens"].map(
    lambda tokens: [token for token in tokens if token.replace("_", "").isalpha()]
)
dataframe["processed_text"] = dataframe["tokens"].map(" ".join)
dataframe[["normalized_text", "processed_text"]].head()

,normalized_text,processed_text
0,this study examines the relationship between e...,this study examines the relationship between e...
1,the project started in 2018 following a use-ca...,the project started in following a approach ie...
2,real estate has always been a significant inve...,real estate has always been a significant inve...
3,global policy organizations have emphasized ca...,global policy organizations have emphasized pr...
5,coffee is arguably the most popular drink in t...,coffee is arguably the most popular drink in t...


In [15]:
# ¿Cómo se transforma la lista de tokens en una matriz documento–término apta para analizar el corpus?

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(min_df=2)
document_term_matrix = vectorizer.fit_transform(dataframe["processed_text"])
vocabulary = pd.DataFrame(
    {
        "token": vectorizer.get_feature_names_out(),
        "document_frequency": (document_term_matrix > 0).sum(axis=0).A1,
    }
)
document_term_matrix.shape, vocabulary.head()

((385, 3863),
           token  document_frequency
 0  abbreviation                   2
 1       ability                  15
 2          able                   9
 3         about                  45
 4         above                   6)

In [16]:
# Se guardan abstracts procesados, vocabulario y matriz para conservar una representación reproducible del corpus.

import json
from pathlib import Path
from scipy import sparse

submission_dir = Path("../submission")
dataframe.drop(columns="tokens").to_csv(submission_dir / "tokenized_abstracts.csv", index=False)
vocabulary.to_csv(submission_dir / "vocabulary.csv", index=False)
sparse.save_npz(submission_dir / "document_term_matrix.npz", document_term_matrix)
with (submission_dir / "matrix_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "documents": document_term_matrix.shape[0],
            "terms": document_term_matrix.shape[1],
            "minimum_document_frequency": 2,
            "minimum_abstract_words": minimum_abstract_words,
            "excluded_abstracts": excluded_abstracts,
            "copyright_notices_removed": int(dataframe["has_copyright_notice"].sum()),
        },
        file,
        indent=2,
    )